# 答辩复现：ROS 2 视觉与 UART

检查文件 → 启动 EKF → 看 RViz 和 ROS 消息 → 停止 EKF → 跑 UART 测试。

ROS 2 节点是运行中的程序；话题是程序交换数据的通道；RViz2 是查看图像和三维标记的窗口。

EKF 主线和 UART 已完成本机实测，重投影问题已修复。下面按步骤演示各项；坐标变换审计的说明见对应章节。

## 0. 找到仓库并检查材料

In [1]:
from pathlib import Path
import os, signal, shlex, shutil, subprocess, sys, tempfile, time

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "ros2_ws" / "build_ekf.sh").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("没找到仓库根目录。请在仓库根目录打开 notebook 后重试。")
MODEL = ROOT / "01_神经网络/yolo_pose/final_model/best.onnx"
VIDEO_TEST = ROOT / "00_题目原件/videos/test.mp4"
VIDEO_ARMORS = ROOT / "00_题目原件/videos/armors_yolo_test.mp4"
ROS_SETUP = Path("/opt/ros/lyrical/setup.bash")
RUNTIME_SETUP = Path.home() / "armor_ekf_runtime/install/setup.bash"
print("仓库：", ROOT)
for label, path in [("ROS 2 环境", ROS_SETUP), ("模型", MODEL), ("EKF/坐标视频", VIDEO_TEST), ("神经网络视频", VIDEO_ARMORS)]:
    print(("[找到] " if path.is_file() else "[缺少] ") + label + ": " + str(path))

仓库： /home/seb/桌面/2026赛季视觉组考核资料
[找到] ROS 2 环境: /opt/ros/lyrical/setup.bash
[找到] 模型: /home/seb/桌面/2026赛季视觉组考核资料/01_神经网络/yolo_pose/final_model/best.onnx
[找到] EKF/坐标视频: /home/seb/桌面/2026赛季视觉组考核资料/00_题目原件/videos/test.mp4
[找到] 神经网络视频: /home/seb/桌面/2026赛季视觉组考核资料/00_题目原件/videos/armors_yolo_test.mp4


## 1. 首次构建

In [5]:
if not ROS_SETUP.is_file():
    raise FileNotFoundError(f"没有找到 ROS 环境：{ROS_SETUP}")
build = subprocess.run(["bash", "ros2_ws/build_ekf.sh"], cwd=ROOT, text=True)
if build.returncode != 0:
    raise RuntimeError("构建失败。查看上方第一条 error，并确认 ROS 依赖已安装、网络可用。")
print("构建成功。")

Starting >>> armor_interfaces
Starting >>> armor_video_publisher
Finished <<< armor_interfaces [0.95s]
Finished <<< armor_video_publisher [2.09s]
Starting >>> armor_ekf
Finished <<< armor_ekf [0.22s]

Summary: 3 packages finished [2.47s]
Load the workspace with: source /home/seb/armor_ekf_runtime/install/setup.bash
构建成功。


## 2. 为后续 ROS 命令加载环境

本单元为 notebook 后续启动的每个 ROS 命令加载 ROS 和工作区环境。看到“环境已加载”再继续。

In [6]:
def make_ros_env():
    if not ROS_SETUP.is_file() or not RUNTIME_SETUP.is_file():
        raise FileNotFoundError("ROS 或工作区 install/setup.bash 不存在；请先完成第 1 节构建。")
    setup = f"source {shlex.quote(str(ROS_SETUP))} && source {shlex.quote(str(RUNTIME_SETUP))} && env -0"
    result = subprocess.run(["bash", "-c", setup], cwd=ROOT, capture_output=True, check=True)
    env = {}
    for item in result.stdout.split(b"\0"):
        if item and b"=" in item:
            key, value = item.split(b"=", 1)
            env[key.decode()] = value.decode(errors="replace")
    return env

ROS_ENV = make_ros_env()
print("ROS 环境已加载；ros2 命令位置：", shutil.which("ros2", path=ROS_ENV.get("PATH")))

ROS 环境已加载；ros2 命令位置： /opt/ros/lyrical/bin/ros2


In [7]:
# 通用小工具：运行短 ROS 命令、启动/停止持续运行的演示程序。
def ros(args, timeout=20):
    result = subprocess.run(["ros2", *args], cwd=ROOT, env=ROS_ENV, text=True,
                            capture_output=True, timeout=timeout)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="", file=sys.stderr)
    if result.returncode:
        raise RuntimeError(f"ros2 命令退出码 {result.returncode}: {args}")

DEMO = None
DEMO_LOG = None
DEMO_LOG_HANDLE = None
RVIZ = None
RVIZ_LOG = None
RVIZ_LOG_HANDLE = None
def start_demo(launch_file, video=VIDEO_TEST, fps="20.0", loop="true"):
    global DEMO, DEMO_LOG, DEMO_LOG_HANDLE
    if DEMO is not None and DEMO.poll() is None:
        raise RuntimeError("当前 ROS 演示还在运行。先运行停止单元，再启动下一项。")
    if not MODEL.is_file() or not video.is_file():
        raise FileNotFoundError("模型或所选视频不存在；先重新运行第 0 节检查。")
    DEMO_LOG = Path(tempfile.gettempdir()) / f"defense_{launch_file.replace('.launch.py', '')}.log"
    DEMO_LOG_HANDLE = open(DEMO_LOG, "w")
    command = ["ros2", "launch", "armor_ekf", launch_file, f"model_path:={MODEL}",
               f"video_path:={video}", f"fps:={fps}", f"loop:={loop}"]
    DEMO = subprocess.Popen(command, cwd=ROOT, env=ROS_ENV, stdout=DEMO_LOG_HANDLE,
                            stderr=subprocess.STDOUT, start_new_session=True)
    time.sleep(3)
    if DEMO.poll() is not None:
        print(DEMO_LOG.read_text(errors="replace"))
        raise RuntimeError("ROS 节点提前退出；请看上方日志找第一条 error。")
    print(f"已启动 {launch_file}。保持它运行；下一步检查话题或打开对应 RViz。")
    print("启动日志：", DEMO_LOG)

def stop_demo():
    global DEMO, DEMO_LOG_HANDLE
    if DEMO is None or DEMO.poll() is not None:
        print("当前没有运行中的 ROS 演示。")
        return
    try:
        os.killpg(DEMO.pid, signal.SIGINT)
        DEMO.wait(timeout=8)
    except subprocess.TimeoutExpired:
        os.killpg(DEMO.pid, signal.SIGTERM)
        DEMO.wait(timeout=5)
    if DEMO_LOG_HANDLE:
        DEMO_LOG_HANDLE.close()
        DEMO_LOG_HANDLE = None
    print("ROS 演示已停止。")
    if DEMO_LOG and DEMO_LOG.exists():
        print("最后几行日志：\n" + "\n".join(DEMO_LOG.read_text(errors="replace").splitlines()[-8:]))

def start_rviz(config):
    global RVIZ, RVIZ_LOG, RVIZ_LOG_HANDLE
    if RVIZ is not None and RVIZ.poll() is None:
        print("RViz2 已经打开。")
        return
    RVIZ_LOG = Path(tempfile.gettempdir()) / f"defense_{config}.rviz.log"
    RVIZ_LOG_HANDLE = open(RVIZ_LOG, "w")
    RVIZ = subprocess.Popen(["rviz2", "-d", str(ROOT / "ros2_ws/rviz" / config)],
                            cwd=ROOT, env=ROS_ENV, stdout=RVIZ_LOG_HANDLE,
                            stderr=subprocess.STDOUT, start_new_session=True)
    time.sleep(3)
    if RVIZ.poll() is not None:
        print(RVIZ_LOG.read_text(errors="replace"))
        raise RuntimeError("RViz2 没有正常启动。确认在本机桌面环境运行，而不是无图形界面的远程 kernel。")
    print("RViz2 已启动。切换到 RViz 窗口查看画面。关闭时运行下一格停止 RViz。")

def stop_rviz():
    global RVIZ, RVIZ_LOG_HANDLE
    if RVIZ is None or RVIZ.poll() is not None:
        print("当前没有运行中的 RViz2。")
        return
    try:
        os.killpg(RVIZ.pid, signal.SIGINT)
        RVIZ.wait(timeout=5)
    except subprocess.TimeoutExpired:
        os.killpg(RVIZ.pid, signal.SIGTERM)
        RVIZ.wait(timeout=3)
    if RVIZ_LOG_HANDLE:
        RVIZ_LOG_HANDLE.close()
        RVIZ_LOG_HANDLE = None
    print("RViz2 已关闭。")

## 3. EKF：相机坐标系下的观测角与滤波角

运行下一格后，节点会持续播放 `test.mp4`。正常情况下单元格会显示“已启动”，不会一直占住 notebook。

In [8]:
start_demo("ekf.launch.py", video=VIDEO_TEST)

已启动 ekf.launch.py。保持它运行；下一步检查话题或打开对应 RViz。
启动日志： /tmp/defense_ekf.log


保持 EKF 节点运行。运行下一格查看输入图像、算法输出和自定义消息。RViz2 的窗口由后面的单元格打开。

In [9]:
print("当前装甲板相关话题：")
ros(["topic", "list"])
print("一条 EKF 结果消息：")
ros(["topic", "echo", "/armor/yaw", "--once"])
print("自定义消息字段：")
ros(["interface", "show", "armor_interfaces/msg/YawEstimate"])

当前装甲板相关话题：
/armor/ekf/image_raw
/armor/image_raw
/armor/yaw
/parameter_events
/rosout
一条 EKF 结果消息：
header:
  stamp:
    sec: 1791891144
    nanosec: 34975229
  frame_id: camera_optical_frame
detection_valid: true
measurement_used: true
filter_initialized: true
target_reset: false
track_id: 1
visual_track_id: 1
target_switched: false
class_id: 2
detection_confidence: 0.8184695243835449
observed_yaw: -0.07963887878102105
filtered_yaw: -0.08049750687481882
yaw_rate: 0.0023084904610318323
yaw_variance: 0.003801665136060721
reprojection_error: 0.09161643094489991
pose:
  position:
    x: 0.06954249845522299
    y: 0.08339641706961877
    z: 4.408871307717037
  orientation:
    x: 0.5510778731174754
    y: 0.5884202889353753
    z: -0.43784261929155405
    w: 0.397955502614853
corners:
- x: 685.12646484375
  y: 611.4774780273438
  z: 0.0
- x: 685.6967163085938
  y: 653.4760131835938
  z: 0.0
- x: 794.6762084960938
  y: 654.8516235351562
  z: 0.0
- x: 794.4071655273438
  y: 612.414794921875
 

In [10]:
start_rviz("armor_ekf.rviz")

RViz2 已启动。切换到 RViz 窗口查看画面。关闭时运行下一格停止 RViz。


切换到 RViz2 窗口。正常时会看到带有装甲板角点和姿态文字的图像。看完后运行下面两个单元格停止 RViz 和 EKF。

In [11]:
stop_rviz()

当前没有运行中的 RViz2。


In [12]:
stop_demo()

ROS 演示已停止。
最后几行日志：
[INFO] [armor_ekf_node-1]: process started with pid [197791]
[armor_ekf_node-1] [INFO] [1791882927.796652526] [armor_ekf]: integrated video source publishing and processing /home/seb/桌面/2026赛季视觉组考核资料/00_题目原件/videos/test.mp4 on /armor/image_raw at 20.00 FPS
[armor_ekf_node-1] [INFO] [1791882927.796855782] [armor_ekf]: YOLO Pose + IPPE PnP + circular EKF ready; yaw frame: camera_optical_frame; model: /home/seb/桌面/2026赛季视觉组考核资料/01_神经网络/yolo_pose/final_model/best.onnx
[WARNING] [launch]: user interrupted with ctrl-c (SIGINT)
[INFO] [armor_ekf_node-1]: sending signal 'SIGINT' to process[armor_ekf_node-1]
[armor_ekf_node-1] [INFO] [1791882997.714969119] [rclcpp]: signal_handler(signum=2)
[armor_ekf_node-1] [INFO] [1791882997.717448404] [rclcpp]: signal_handler(signum=2)
[INFO] [armor_ekf_node-1]: process has finished cleanly [pid 197791]


## 4. 坐标变换：把相机姿态转换到 odom

先确认上一节 EKF 已停止。运行后看 `/armor/world_yaw`，它的消息坐标系应为 `odom`。

In [13]:
start_demo("world.launch.py", video=VIDEO_TEST)

已启动 world.launch.py。保持它运行；下一步检查话题或打开对应 RViz。
启动日志： /tmp/defense_world.log


In [14]:
ros(["topic", "echo", "/armor/world_yaw", "--once"])
ros(["topic", "echo", "/armor/world_markers", "--once"])
start_rviz("armor_world.rviz")

header:
  stamp:
    sec: 1791883040
    nanosec: 938845088
  frame_id: odom
detection_valid: true
measurement_used: true
filter_initialized: true
target_reset: false
track_id: 1
visual_track_id: 1
target_switched: false
class_id: 2
detection_confidence: 0.8207790851593018
observed_yaw: 2.9912710430114635
filtered_yaw: 2.981344131685517
yaw_rate: 0.06821955680994807
yaw_variance: 0.0032818613652835144
reprojection_error: 0.13292505662876405
pose:
  position:
    x: 0.48407469816623533
    y: -0.07010765848244381
    z: -4.447588355034207
  orientation:
    x: 0.5154188551195956
    y: 0.03177199989035878
    z: 0.8560601998723337
    w: 0.022245853652143772
corners:
- x: 685.3042602539062
  y: 613.2399291992188
  z: 0.0
- x: 686.422607421875
  y: 654.6475830078125
  z: 0.0
- x: 795.3106689453125
  y: 654.2637329101562
  z: 0.0
- x: 794.116943359375
  y: 612.7316284179688
  z: 0.0
---
markers:
- header:
    stamp:
      sec: 1791883042
      nanosec: 513635831
    frame_id: odom
  ns: a

在 RViz 看完后，按顺序运行下面两格停止 RViz 和坐标变换节点。

本机节点可启动并发布结果；严格审计脚本曾漏收到一帧输入图像时间戳。现场展示话题和 RViz 即可，不要把该审计脚本说成此次现场通过项。

In [17]:
stop_rviz()

当前没有运行中的 RViz2。


In [ ]:
stop_demo()

## 5. 重投影：显示四块装甲板模型

重投影问题已修复。本节按题目要求使用 `test.mp4`。

运行下一格启动节点，再运行话题检查和 RViz 单元格。观察 `/armor/reproject/image_raw` 的标注图、`/armor/reproject/poses` 的装甲板姿态，以及 `/armor/reproject/markers` 的三维模型。

在 RViz 中确认图像显示重投影框，并检查四块装甲板模型与机器人运动相符。模型包含四块装甲板；图像中的实际可见性取决于朝向、遮挡和投影位置。话题输出若暂时为空，等待视频中出现有效识别后重新运行检查单元格。

In [31]:
start_demo("reprojection.launch.py", video=VIDEO_TEST)

已启动 reprojection.launch.py。保持它运行；下一步检查话题或打开对应 RViz。
启动日志： /tmp/defense_reprojection.log


In [32]:
ros(["topic", "echo", "/armor/reproject/poses", "--once"])
ros(["topic", "echo", "/armor/reproject/markers", "--once"])
start_rviz("armor_reprojection.rviz")

header:
  stamp:
    sec: 1791884434
    nanosec: 932214320
  frame_id: odom
poses: []
---
markers:
- header:
    stamp:
      sec: 1791884437
      nanosec: 378677174
    frame_id: odom
  ns: four_armor_model
  id: 0
  type: 1
  action: 2
  pose:
    position:
      x: 0.0
      y: 0.0
      z: 0.0
    orientation:
      x: 0.0
      y: 0.0
      z: 0.0
      w: 1.0
  scale:
    x: 0.01
    y: 0.135
    z: 0.055
  color:
    r: 1.0
    g: 0.699999988079071
    b: 0.0
    a: 1.0
  lifetime:
    sec: 0
    nanosec: 300000000
  frame_locked: false
  points: []
  colors: []
  texture_resource: ''
  texture:
    header:
      stamp:
        sec: 0
        nanosec: 0
      frame_id: ''
    format: ''
    data: []
  uv_coordinates: []
  text: ''
  mesh_resource: ''
  mesh_file:
    filename: ''
    data: []
  mesh_use_embedded_materials: false
- header:
    stamp:
      sec: 1791884437
      nanosec: 378677174
    frame_id: odom
  ns: four_armor_model
  id: 1
  type: 1
  action: 2
  pose:
  

查看后运行下面两格停止 RViz 和 ROS 节点。

In [33]:
stop_rviz()

当前没有运行中的 RViz2。


In [34]:
stop_demo()

ROS 演示已停止。
最后几行日志：
[INFO] [armor_ekf_node-1]: process started with pid [202033]
[armor_ekf_node-1] [INFO] [1791884431.286324330] [armor_ekf]: integrated video source publishing and processing /home/seb/桌面/2026赛季视觉组考核资料/00_题目原件/videos/test.mp4 on /armor/image_raw at 20.00 FPS
[armor_ekf_node-1] [INFO] [1791884431.286575740] [armor_ekf]: YOLO Pose + IPPE PnP + circular EKF ready; yaw frame: odom; model: /home/seb/桌面/2026赛季视觉组考核资料/01_神经网络/yolo_pose/final_model/best.onnx
[WARNING] [launch]: user interrupted with ctrl-c (SIGINT)
[INFO] [armor_ekf_node-1]: sending signal 'SIGINT' to process[armor_ekf_node-1]
[armor_ekf_node-1] [INFO] [1791884461.466844017] [rclcpp]: signal_handler(signum=2)
[armor_ekf_node-1] [INFO] [1791884461.470567862] [rclcpp]: signal_handler(signum=2)
[INFO] [armor_ekf_node-1]: process has finished cleanly [pid 202033]


## 6. UART：协议单测与官方虚拟串口回环

In [3]:
uart_test = subprocess.run(["/usr/bin/python3", "test_protocol.py", "-v"],
                           cwd=ROOT / "02_UART/uart", text=True, capture_output=True)
print(uart_test.stdout)
if uart_test.stderr:
    print(uart_test.stderr, file=sys.stderr)
if uart_test.returncode:
    raise RuntimeError("UART 单元测试失败。")
print("UART 单元测试通过。")


UART 单元测试通过。


test_bad_tail_resynchronization (__main__.ProtocolTests.test_bad_tail_resynchronization) ... ok
test_every_split_position (__main__.ProtocolTests.test_every_split_position) ... ok
test_noise_bad_padding_and_concatenated_packets (__main__.ProtocolTests.test_noise_bad_padding_and_concatenated_packets) ... ok
test_signed_boundaries (__main__.ProtocolTests.test_signed_boundaries) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.001s

OK



然后运行官方回环。下一格会先打开测试器，再等待不到 1 秒启动回传程序；测试器本身只等待约 5 秒。成功标志是四组数据都回传一致，测试器显示 4/4 通过。

In [4]:
tester = subprocess.Popen([str(ROOT / "02_UART/virtual_serial_tester")], cwd=ROOT,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    time.sleep(0.7)
    echo = subprocess.run(["/usr/bin/python3", "uart_echo.py", "--count", "4"],
                         cwd=ROOT / "02_UART/uart", text=True, capture_output=True, timeout=15)
    print("--- 回传程序 ---\n" + echo.stdout)
    if echo.stderr:
        print(echo.stderr, file=sys.stderr)
    tester_output, _ = tester.communicate(timeout=10)
    print("--- 官方测试器 ---\n" + tester_output)
    if echo.returncode != 0 or tester.returncode != 0:
        raise RuntimeError("UART 回环失败；确认测试器和回传程序都正常启动后再重试。")
    print("UART 官方回环通过。")
except Exception:
    if tester.poll() is None:
        tester.kill()
        tester.wait()
    raise

--- 回传程序 ---
Opened /tmp/test_com: 115200 baud, 8N1
[1] uint8=10 int16_be=256 int16_le=1 echoed
[2] uint8=255 int16_be=-1 int16_le=-256 echoed
[3] uint8=0 int16_be=0 int16_le=0 echoed
[4] uint8=127 int16_be=32767 int16_le=-32768 echoed

--- 官方测试器 ---
========== 考核环境已启动 ==========
>> 虚拟串口路径: /dev/pts/6
>> 固定链接路径: /tmp/test_com
>> 请运行你的被测程序连接上述串口...
>> 等待被测程序连接（5秒）...
>> 在此期间启动程序连接

[测试 1/4] 正数：u8=0x0A, BE=0x0100(256), LE=0x0001(1)
  发送包:   [0]=FF(头) [1]=0A(u8) [2..3]=0100(BE:256) [4..5]=0100(LE:1) [6..14]=填充 [15]=0D(尾)
发送完成
  收到回应:   [0]=FF(头) [1]=0A(u8) [2..3]=0100(BE:256) [4..5]=0100(LE:1) [6..14]=填充 [15]=0D(尾)
  [通过] 回应包内容完全一致

[测试 2/4] 负数：u8=0xFF, BE=-1(0xFFFF), LE=-256(0xFF00)
  发送包:   [0]=FF(头) [1]=FF(u8) [2..3]=FFFF(BE:-1) [4..5]=00FF(LE:-256) [6..14]=填充 [15]=0D(尾)
发送完成
  收到回应:   [0]=FF(头) [1]=FF(u8) [2..3]=FFFF(BE:-1) [4..5]=00FF(LE:-256) [6..14]=填充 [15]=0D(尾)
  [通过] 回应包内容完全一致

[测试 3/4] 零值：u8=0x00, BE=0, LE=0
  发送包:   [0]=FF(头) [1]=00(u8) [2..3]=0000(BE:0) [4..5]=0000(LE:0) [6..